# N044 · 旋转数组、矩阵与峰值

**目标：** 根据局部结构构造正确的舍弃条件。

**先修：** N018, N042。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 旋转有序；重复导致退化；矩阵单调；峰值存在性；二维转索引。

## 从问题到算法

旋转数组至少有一半有序，可用目标是否落在有序半区确定方向；有重复时端点相等可能无法判断，只能缩小边界，因此最坏退化线性。矩阵二分需要跨行也递增，不能只凭每行有序。峰值搜索依赖相邻不相等和边界视为负无穷的契约。

## 最小实现

**本章接口：** `search_rotated_unique(nums, target)`、`search_rotated_with_duplicates(nums, target)`、`search_matrix(matrix, target)`、`find_peak(nums)`

In [1]:
def search_rotated_unique(nums,target):
    lo,hi=0,len(nums)-1
    while lo<=hi:
        mid=(lo+hi)//2
        if nums[mid]==target: return mid
        if nums[lo]<=nums[mid]:
            if nums[lo]<=target<nums[mid]: hi=mid-1
            else: lo=mid+1
        else:
            if nums[mid]<target<=nums[hi]: lo=mid+1
            else: hi=mid-1
    return -1

def search_rotated_with_duplicates(nums,target):
    lo,hi=0,len(nums)-1
    while lo<=hi:
        mid=(lo+hi)//2
        if nums[mid]==target: return True
        if nums[lo]==nums[mid]==nums[hi]: lo+=1; hi-=1
        elif nums[lo]<=nums[mid]:
            if nums[lo]<=target<nums[mid]: hi=mid-1
            else: lo=mid+1
        else:
            if nums[mid]<target<=nums[hi]: lo=mid+1
            else: hi=mid-1
    return False

def search_matrix(matrix,target):
    if not matrix or not matrix[0]: return False
    rows,cols=len(matrix),len(matrix[0]); lo,hi=0,rows*cols
    while lo<hi:
        mid=(lo+hi)//2; value=matrix[mid//cols][mid%cols]
        if value<target: lo=mid+1
        else: hi=mid
    return lo<rows*cols and matrix[lo//cols][lo%cols]==target

def find_peak(nums):
    if not nums: raise ValueError('requires a nonempty sequence without equal neighbors')
    lo,hi=0,len(nums)-1
    while lo<hi:
        mid=(lo+hi)//2
        if nums[mid]<nums[mid+1]: lo=mid+1
        else: hi=mid
    return lo

## 正确性、前提与复杂度

无重复旋转数组的一半保有整体次序，区间成员测试可安全排除另一半；重复版本只在确实无法判别时去端点。峰值若中部上升，右半必在某处转降或在右边界形成峰。

**代价：** 无重复搜索、矩阵查找、峰值 O(log n)；重复搜索最坏 O(n)。均O(1)辅助空间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

base=[0,1,2,3,4]
show_table(['旋转量','数组','目标2的位置'],[(k,base[k:]+base[:k],search_rotated_unique(base[k:]+base[:k],2)) for k in range(5)])

旋转量,数组,目标2的位置
0,"[0, 1, 2, 3, 4]",2
1,"[1, 2, 3, 4, 0]",1
2,"[2, 3, 4, 0, 1]",0
3,"[3, 4, 0, 1, 2]",4
4,"[4, 0, 1, 2, 3]",3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from itertools import combinations_with_replacement,product
for n in range(1,7):
    base=list(range(n))
    for k in range(n):
        a=base[k:]+base[:k]
        for t in range(-1,n+1): assert search_rotated_unique(a,t)==(a.index(t) if t in a else -1)
    for base in combinations_with_replacement([0,1,2],n):
        for k in range(n):
            a=base[k:]+base[:k]
            for t in range(4): assert search_rotated_with_duplicates(a,t)==(t in a)
for a in product(range(3),repeat=5):
    if all(a[i]!=a[i+1] for i in range(4)):
        p=find_peak(a); assert (p==0 or a[p]>a[p-1]) and (p==4 or a[p]>a[p+1])
assert search_matrix([[1,3,5],[7,9,11]],9)
print('N044: 所有本章断言通过')

N044: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 处理旋转数组重复值。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较74的全局有序与240的行列有序。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 33. Search in Rotated Sorted Array（canonical）
- 81. Search in Rotated Sorted Array II（canonical）
- 74. Search a 2D Matrix（canonical）
- 162. Find Peak Element（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。